# Audio Captions (CLAP-Cap) + LLM Rewrite

Generates **audio captions** for every WAV segment using [CLAP-Cap](https://github.com/microsoft/CLAP)
(`msclap`, `version='clapcap'`), then fuses them with the video captions and Whisper transcripts
from `text.ipynb` into clean AV descriptions using Claude Haiku.

Outputs:
```
outputs/model_embeddings/text/{seg}s/audio_captions.json
outputs/model_embeddings/text/{seg}s/rewritten_text_inputs.json
```

**Kernel:** `audiocaption`  
**Run order:** run `text.ipynb` first (generates `captions.json` + `transcripts.json`) before the rewrite step.

CLAP-Cap weights auto-download to `~/.cache/msclap` on first run (~1 GB).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from paths import DATA, OUTPUTS

import gc
import json
import re
from pathlib import Path

import torch
from natsort import natsorted

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# -- CONFIG -------------------------------------------------------------------
CLAP_BATCH_SIZE = 16   # reduce if OOM; CLAP-Cap is lightweight

# BIN_SEC: window duration; SKIP_SEC: stride (= BIN_SEC → non-overlapping)
BIN_SEC  = 5.0
SKIP_SEC = 5.0

# Each tuple: (bin_sec, skip_sec) — output dir: bin{N}s_skip{N}s
SEGMENT_DURATIONS = [(5.0, 5.0), (10.0, 10.0)]

DATA_BASE   = DATA / "segmented_stimulus/filtered"
OUTPUT_BASE = OUTPUTS / "model_embeddings"
TEXT_OUT    = OUTPUT_BASE / "text"
# -----------------------------------------------------------------------------


def find_chunk_pairs(data_base: Path, bin_sec: float, skip_sec: float = None) -> list[tuple[Path, Path]]:
    """Return sorted (av_mp4, av_mp4) pairs from combined AV chunk dirs.

    Looks for {stem}_av_chunks_{dur}s or {stem}_av_chunks_{dur}s_skip{skip}s.
    Both tuple elements point to the same .mp4 (video + audio from one file).
    """
    if skip_sec is None:
        skip_sec = bin_sec
    dur_int  = int(bin_sec)
    skip_int = int(skip_sec)
    chunk_suffix = (f"_av_chunks_{dur_int}s" if skip_int == dur_int
                    else f"_av_chunks_{dur_int}s_skip{skip_int}s")
    pairs = []
    for vid_subdir in natsorted(data_base.glob("Video*")):
        m = re.search(r"Video(\d+)$", vid_subdir.name)
        if not m:
            continue
        chunk_dir = vid_subdir / f"{vid_subdir.name}{chunk_suffix}"
        if not chunk_dir.exists():
            continue
        for av in natsorted(chunk_dir.glob("*_part_*.mp4")):
            pairs.append((av, av))
    return pairs


print("Chunk pair counts per segment duration:")
for _b, _s in SEGMENT_DURATIONS:
    _p = find_chunk_pairs(DATA_BASE, _b, _s)
    print(f"  bin{int(_b)}s_skip{int(_s)}s -> {len(_p):4d} pairs")

## Audio Captions — CLAP-Cap (msclap)

`msclap` with `version='clapcap'` — Microsoft's CLAP language–audio pre-training model with a captioning head.
Weights auto-download to `~/.cache/msclap` on first run (~1 GB).

Output: `outputs/model_embeddings/text/{seg}s/audio_captions.json`

In [ ]:
from msclap import CLAP

clap_model = CLAP(version="clapcap", use_cuda=(DEVICE == "cuda"))
print("CLAP-Cap loaded.")


@torch.no_grad()
def generate_audio_captions(aud_paths: list, batch_size: int = CLAP_BATCH_SIZE) -> list:
    results  = []
    cur_size = batch_size
    i = 0
    while i < len(aud_paths):
        batch = [str(p) for p in aud_paths[i : i + cur_size]]
        try:
            caps = clap_model.generate_caption(batch)
            results.extend(caps)
            i += len(batch)
            print(f"  {i}/{len(aud_paths)}", end="\r")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            if cur_size == 1:
                raise
            cur_size = max(1, cur_size // 2)
            print(f"\n  OOM — reducing batch size to {cur_size}")
    print()
    return results

In [ ]:
for bin_sec, skip_sec in SEGMENT_DURATIONS:
    out_dir  = TEXT_OUT / f"bin{int(bin_sec)}s_skip{int(skip_sec)}s"
    out_path = out_dir / "audio_captions.json"

    if out_path.exists():
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- audio_captions.json already exists, skipping")
        continue

    pairs = find_chunk_pairs(DATA_BASE, bin_sec, skip_sec)
    if not pairs:
        print(f"  bin{int(bin_sec)}s_skip{int(skip_sec)}s -- no pairs found, skipping")
        continue

    print(f"\nbin{int(bin_sec)}s_skip{int(skip_sec)}s -- audio-captioning {len(pairs)} segments ...")
    audio_captions = generate_audio_captions([p[1] for p in pairs])

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(audio_captions, ensure_ascii=False, indent=2))
    print(f"  Saved: {out_path}  ({len(audio_captions)} entries)")
    if audio_captions:
        print(f"  Sample [0]: {audio_captions[0][:120]}")

In [ ]:
del clap_model
torch.cuda.empty_cache()
gc.collect()
print("CLAP-Cap unloaded.")

## LLM Rewrite — Claude Haiku

Fuses three inputs per segment into one clean AV sentence:
- `captions.json` — visual captions from InternVL2.5-8B (`text.ipynb`)
- `audio_captions.json` — audio captions from CLAP-Cap (above)
- `transcripts.json` — Whisper speech transcripts (`text.ipynb`)

**Prereq:** run `text.ipynb` first to generate `captions.json` and `transcripts.json`.  
Set `ANTHROPIC_API_KEY` before running:
```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

Output: `outputs/model_embeddings/text/{seg}s/rewritten_text_inputs.json`

In [ ]:
import os
import time
import anthropic

REWRITE_MODEL      = "claude-haiku-4-5-20251001"
REWRITE_RATE_DELAY = 0.05   # seconds between API calls; raise if rate-limited

_SYSTEM = """\
You are an audiovisual caption editor. You receive three inputs about a short video clip:
  Video caption : what is visually happening (from a video captioner)
  Audio caption : what sounds are present (from an audio captioner)
  Transcript    : words spoken, if any (from Whisper speech-to-text — may be empty or noisy)

Your task: write EXACTLY ONE clean sentence that captures both the visual scene and its
acoustic character.

Rules (follow exactly):
1. Include ONLY information present in the inputs — do not add, infer, or hallucinate.
2. If the transcript contains hallucinations (song lyrics not matching the scene, repetitive
   filler phrases, or random words when the audio caption indicates no speech), discard it.
3. If the transcript contains real speech relevant to the scene, incorporate its gist.
4. Merge the visual and acoustic perspectives naturally.
5. Present tense, third-person, factual. No hedging.
6. Output the sentence only — no preamble or explanation."""

api_key = os.environ.get("ANTHROPIC_API_KEY", "")
if not api_key:
    raise SystemExit("ANTHROPIC_API_KEY not set. Run: export ANTHROPIC_API_KEY='sk-ant-...'")
client = anthropic.Anthropic(api_key=api_key)
print(f"Anthropic client ready. Model: {REWRITE_MODEL}")

In [ ]:
def _is_hallucinated_transcript(transcript: str, audio_caption: str) -> bool:
    t = transcript.strip()
    if not t:
        return False
    words = t.split()
    if len(words) >= 6:
        phrases = [" ".join(words[i : i + 3]) for i in range(0, len(words) - 2, 3)]
        if len(set(phrases)) <= 2:
            return True
    ac_lower = audio_caption.lower()
    if any(kw in ac_lower for kw in ["music", "instrumental", "melody", "song", "tune"]):
        if len(words) < 20:
            return True
    return False


def rewrite_one(video_cap: str, audio_cap: str, transcript: str) -> str:
    trn = transcript.strip()
    if _is_hallucinated_transcript(trn, audio_cap):
        trn = ""
    trn_display = trn if trn else "(no speech)"
    resp = client.messages.create(
        model=REWRITE_MODEL,
        max_tokens=120,
        system=_SYSTEM,
        messages=[{"role": "user", "content": (
            f"Video caption: {video_cap}\n"
            f"Audio caption: {audio_cap}\n"
            f"Transcript: {trn_display}\n\n"
            "Write one clean audiovisual sentence."
        )}],
    )
    return resp.content[0].text.strip()

In [ ]:
import random

for bin_sec, skip_sec in SEGMENT_DURATIONS:
    out_dir = TEXT_OUT / f"bin{int(bin_sec)}s_skip{int(skip_sec)}s"
    rw_path = out_dir / "rewritten_text_inputs.json"

    if rw_path.exists():
        print(f"bin{int(bin_sec)}s_skip{int(skip_sec)}s -- rewritten_text_inputs.json already exists, skipping")
        continue

    cap_path = out_dir / "captions.json"
    ac_path  = out_dir / "audio_captions.json"
    trn_path = out_dir / "transcripts.json"

    missing = [p.name for p in [cap_path, ac_path, trn_path] if not p.exists()]
    if missing:
        print(f"bin{int(bin_sec)}s_skip{int(skip_sec)}s -- MISSING: {missing}  (skipping)")
        continue

    captions       = json.loads(cap_path.read_text())
    audio_captions = json.loads(ac_path.read_text())
    transcripts    = json.loads(trn_path.read_text())

    assert len(captions) == len(audio_captions) == len(transcripts), (
        f"bin{int(bin_sec)}s_skip{int(skip_sec)}s: length mismatch  "
        f"caps={len(captions)}  ac={len(audio_captions)}  trn={len(transcripts)}"
    )

    print(f"bin{int(bin_sec)}s_skip{int(skip_sec)}s -- rewriting {len(captions)} segments ...")
    rewritten = []
    for i, (vc, ac, tr) in enumerate(zip(captions, audio_captions, transcripts)):
        rewritten.append(rewrite_one(vc, ac, tr))
        if REWRITE_RATE_DELAY:
            time.sleep(REWRITE_RATE_DELAY)
        if (i + 1) % 50 == 0 or (i + 1) == len(captions):
            print(f"  {i+1}/{len(captions)}", end="\r")
    print(f"  {len(captions)}/{len(captions)} done")

    rw_path.write_text(json.dumps(rewritten, ensure_ascii=False, indent=2))
    print(f"  Saved: {rw_path}\n")

    print("  Sample rewrites:")
    for idx in random.sample(range(len(rewritten)), min(3, len(rewritten))):
        print(f"\n  [{idx}]")
        print(f"  VID  : {captions[idx][:90]}")
        print(f"  AUD  : {audio_captions[idx][:90]}")
        print(f"  TRANS: {transcripts[idx][:70]}")
        print(f"  -> RW: {rewritten[idx]}")
    print()

## Verify outputs

In [ ]:
for bin_sec, skip_sec in SEGMENT_DURATIONS:
    out_dir = TEXT_OUT / f"bin{int(bin_sec)}s_skip{int(skip_sec)}s"
    files = {
        "captions":              out_dir / "captions.json",
        "audio_captions":        out_dir / "audio_captions.json",
        "transcripts":           out_dir / "transcripts.json",
        "rewritten_text_inputs": out_dir / "rewritten_text_inputs.json",
    }
    statuses = {k: ("OK" if p.exists() else "MISSING") for k, p in files.items()}
    ready    = all(v == "OK" for v in statuses.values())
    print(f"bin{int(bin_sec)}s_skip{int(skip_sec)}s  {'READY' if ready else 'NOT ready'}  {statuses}")